# Lab 3 : Semantic Network

## Objectives

1. To understand the concept of Semantic Networks and their role in representing knowledge through nodes and relationships.

2. To study Natural Language Processing (NLP) techniques such as Part-of-Speech (POS) Tagging and Named Entity Recognition (NER).

3. To learn how text data is converted into numerical representations using Vector Embeddings for machine learning applications.

4. To explore Knowledge Triples (Subject–Predicate–Object) and similarity measures used in recommendation systems and semantic analysis.

## Theory :

#### Semantic networks are a knowledge representation technique in AI that connects concepts through relationships to organize information meaningfully and support intelligent reasoning and inference.

* Uses nodes to represent concepts and edges to represent relationships.
* Helps machines understand connections between different ideas.
* Commonly used in NLP and AI reasoning systems.

---

## Part of Speech (POS) Tagging

Parts of Speech (PoS) tagging is a fundamental task in Natural Language Processing (NLP) where each word in a sentence is assigned a grammatical category such as noun, verb, adjective or adverb. This process help machines to understand the structure and meaning of sentences by identifying the roles of words and their relationships.

### Architecture of POS

![pos Diagram](https://media.geeksforgeeks.org/wp-content/uploads/20250819114113512745/architecture_of_pos_tagging.webp)

#### It plays an important role in various NLP applications including machine translation, sentiment analysis and information retrieval, by bridging the gap between human language and machine understanding.

#### Key Concepts in POS Tagging

* Parts of Speech: These are categories like nouns, verbs, adjectives, adverbs, etc that define the role of a word in a sentence.
* Tagging: The process of assigning a specific part-of-speech label to each word in a sentence.
* Corpus: A large collection of text data used to train POS taggers.

#### Example

![pos example ](https://media.geeksforgeeks.org/wp-content/uploads/20250819114114256836/a_quick_brown_fox_jumps_over_a_lazy_dog.webp)

---

## Named Entity Recognition (NER)

Named entity recognition (NER) also called entity chunking or entity extraction is a component of natural language processing (NLP) that identifies predefined categories of objects in a body of text.

Named Entity Recognition (NER) in NLP focuses on identifying and categorizing important information known as entities in text. These entities can be names of people, places, organizations, dates, etc. It helps in transforming unstructured text into structured information which helps in tasks like text summarization, knowledge graph creation and question answering.

### Working of NER
![ner](https://media.geeksforgeeks.org/wp-content/uploads/20251216102233196838/NER.webp)

NER helps in detecting specific information and sort it into predefined categories. It plays an important role in enhancing other NLP tasks like part-of-speech tagging and parsing. 

## Vector Embedding

Vector embedding are digital fingerprints or numerical representations of words or other pieces of data. Each object is transformed into a list of numbers called a vector. These vectors captures properties of the object in a more manageable and understandable form for machine learning models.

![vector e ](https://media.geeksforgeeks.org/wp-content/uploads/20250531164711674381/Vector-Embedding.webp)

#### Use of Vector embedding
* Compare Similarities: Measures vector distances to identify semantically similar data.
* Clustering: Groups related data using algorithms like K-means or DBSCAN.
* Perform Arithmetic Operations: Captures relationships and analogies through vector math.
* Feed Machine Understandable Data: Converts complex data into vectors for machine processing.

## Knowledge Triples

Knowledge graph triples (also called semantic triples or RDF triples) are the atomic building blocks of knowledge graphs, structured as a <subject, predicate, object> triplet that represents a single factual relationship between two entities. 

#### Core Structure and Function
* Atomic Unit: Each triple connects a head entity (subject) to a tail entity (object) via a relation (predicate), forming a directed edge in the graph (e.g., Albert Einstein, bornIn, Ulm).

* Standardization: They follow the Resource Description Framework (RDF) model, where each component is uniquely identifiable via an IRI (Internationalized Resource Identifier), ensuring machine-readable semantic clarity. 

* Graph Representation: A knowledge graph is defined as a multi-relational graph G=(V,R,E), where the set of triplets E constitutes the edges, allowing the representation of heterogeneous data through interconnected nodes and labeled links. 

### Example

![eg](https://media.geeksforgeeks.org/wp-content/uploads/20250605175211527530/Knowledge-Graph.webp)

![eg](https://media.geeksforgeeks.org/wp-content/uploads/20250605175237858173/2.webp)

![eg](https://media.geeksforgeeks.org/wp-content/uploads/20250605175358270217/knowledge-G.webp)

![eg](https://media.geeksforgeeks.org/wp-content/uploads/20250605175446461145/Final-Graph.webp)

## Recomendation system 

### recommender.py

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║         ACADEMIC PAPER RECOMMENDATION ENGINE  v1.0              ║
║         Graph-Based · Citation-Aware · Multi-Strategy           ║
╚══════════════════════════════════════════════════════════════════╝

HOW IT WORKS:
  - Loads network_data.json (generated by lab3.py)
  - Builds a NetworkX directed graph from nodes + edges
  - Runs 3 recommendation strategies:
      1. DIRECT MATCH   — papers directly linked to your topic
      2. CITATION BOOST — papers heavily cited within that topic cluster
      3. CROSS-DOMAIN   — papers bridging your topic with other domains
  - Scores and ranks results, then shows a clean panel UI in terminal
"""

import json
import os
import sys
import networkx as nx
from collections import defaultdict

# ─────────────────────────────────────────────────────────────────
# ANSI COLORS
# ─────────────────────────────────────────────────────────────────
CYAN    = "\033[96m"
PINK    = "\033[95m"
LIME    = "\033[92m"
AMBER   = "\033[93m"
WHITE   = "\033[97m"
GREY    = "\033[90m"
BOLD    = "\033[1m"
DIM     = "\033[2m"
RESET   = "\033[0m"
RED     = "\033[91m"
BLUE    = "\033[94m"

GROUP_COLOR = {
    "Domain":      PINK,
    "Methodology": LIME,
    "Author":      AMBER,
    "Paper":       CYAN,
}

# ─────────────────────────────────────────────────────────────────
# LOAD DATA & BUILD GRAPH
# ─────────────────────────────────────────────────────────────────
def load_graph(path="network_data.json"):
    if not os.path.exists(path):
        print(f"{RED}✗ Cannot find '{path}'. Run lab3.py first to generate it.{RESET}")
        sys.exit(1)

    with open(path) as f:
        data = json.load(f)

    G = nx.DiGraph()

    # Add nodes with metadata
    for node in data["nodes"]:
        G.add_node(node["id"], group=node["group"],
                   category=node.get("category", ""),
                   rec=node.get("rec", ""))

    # Add edges with relationship labels
    for link in data["links"]:
        src = link["source"] if isinstance(link["source"], str) else link["source"]["id"]
        tgt = link["target"] if isinstance(link["target"], str) else link["target"]["id"]
        G.add_edge(src, tgt, label=link["label"])

    return G, data


# ─────────────────────────────────────────────────────────────────
# RECOMMENDATION ENGINE  (3 strategies)
# ─────────────────────────────────────────────────────────────────
def recommend(G, topic_node):
    """
    Returns a list of dicts, each being a recommended paper with a score and reason.
    """
    if topic_node not in G:
        return []

    group = G.nodes[topic_node].get("group", "")
    scores = defaultdict(float)
    reasons = defaultdict(set)

    # ── Strategy 1: DIRECT MATCH ──────────────────────────────────
    # Papers directly connected to the topic via BELONGS_TO / UTILIZED / AUTHORED
    for neighbor in G.predecessors(topic_node):   # edges point paper → topic
        edge_data = G.edges[neighbor, topic_node]
        if G.nodes[neighbor].get("group") == "Paper":
            scores[neighbor] += 3.0
            reasons[neighbor].add(f"Direct {edge_data['label']} link to [{topic_node}]")

    for neighbor in G.successors(topic_node):     # author → paper edges
        edge_data = G.edges[topic_node, neighbor]
        if G.nodes[neighbor].get("group") == "Paper":
            scores[neighbor] += 3.0
            reasons[neighbor].add(f"Direct {edge_data['label']} link from [{topic_node}]")

    # ── Strategy 2: CITATION BOOST ────────────────────────────────
    # Among the direct papers, count how many times each is cited by others in the cluster
    cluster_papers = {n for n, s in scores.items() if s >= 3.0}

    for paper in cluster_papers:
        # Count in-citations FROM other cluster papers
        for citer in G.predecessors(paper):
            if G.edges[citer, paper].get("label") == "CITES":
                citation_boost = 1.5 if citer in cluster_papers else 0.5
                scores[paper] += citation_boost
                reasons[paper].add(f"Cited by {len(list(G.predecessors(paper)))} papers in network")

    # ── Strategy 3: CROSS-DOMAIN BRIDGES ─────────────────────────
    # Papers that belong to a DIFFERENT domain but cite or share methodology with this cluster
    all_paper_nodes = [n for n in G.nodes if G.nodes[n].get("group") == "Paper"]

    for paper in all_paper_nodes:
        if paper in cluster_papers:
            continue
        # Does this paper cite any paper in our cluster?
        for cited in G.successors(paper):
            if cited in cluster_papers and G.edges[paper, cited].get("label") == "CITES":
                scores[paper] += 1.0
                reasons[paper].add(f"Cross-domain bridge: cites [{cited[:40]}...]")
                break

    # ── Filter: only return Paper nodes ───────────────────────────
    results = []
    for paper_id, score in sorted(scores.items(), key=lambda x: -x[1]):
        if G.nodes[paper_id].get("group") != "Paper":
            continue
        # Gather metadata
        domain  = next((G.edges[paper_id, t]["label"] and t for _, t in G.out_edges(paper_id)
                        if G.edges[paper_id, t].get("label") == "BELONGS_TO"), "—")
        method  = next((t for _, t in G.out_edges(paper_id)
                        if G.edges[paper_id, t].get("label") == "UTILIZED"), "—")
        author  = next((s for s, _ in G.in_edges(paper_id)
                        if G.edges[s, paper_id].get("label") == "AUTHORED"), "—")
        rec     = G.nodes[paper_id].get("rec", "")
        citation_count = sum(1 for _ in G.predecessors(paper_id)
                             if G.edges[_, paper_id].get("label") == "CITES")

        results.append({
            "title":    paper_id,
            "score":    round(score, 2),
            "domain":   domain,
            "method":   method,
            "author":   author.replace("_", " "),
            "citations": citation_count,
            "reasons":  list(reasons[paper_id]),
            "rec":      rec,
        })

    return results


# ─────────────────────────────────────────────────────────────────
# TERMINAL UI HELPERS
# ─────────────────────────────────────────────────────────────────
def clear():
    os.system("cls" if os.name == "nt" else "clear")

def banner():
    print(f"""
{CYAN}{BOLD}╔══════════════════════════════════════════════════════════════════╗
║        RESEARCH PAPER RECOMMENDATION ENGINE  v1.0               ║
║        Graph-Based · Citation-Aware · 30 Papers                 ║
╚══════════════════════════════════════════════════════════════════╝{RESET}
""")

def divider(char="─", color=GREY):
    print(f"{color}{char * 68}{RESET}")

def topic_menu(G):
    """Show topic selection panel grouped by type."""
    domains  = sorted([n for n in G.nodes if G.nodes[n]["group"] == "Domain"])
    methods  = sorted([n for n in G.nodes if G.nodes[n]["group"] == "Methodology"])
    authors  = sorted([n for n in G.nodes if G.nodes[n]["group"] == "Author"])

    all_topics = []
    idx = 1

    print(f"{BOLD}{WHITE}  SELECT A TOPIC TO GET PAPER RECOMMENDATIONS{RESET}\n")

    # Domains
    print(f"  {PINK}{BOLD}── DOMAINS ──────────────────────────────────────────────{RESET}")
    for d in domains:
        count = sum(1 for p in G.predecessors(d) if G.nodes[p].get("group") == "Paper")
        print(f"  {GREY}[{WHITE}{BOLD}{idx:2d}{RESET}{GREY}]{RESET}  {PINK}{d:<42}{RESET}  {GREY}{count} papers{RESET}")
        all_topics.append(d)
        idx += 1

    print()

    # Methodologies
    print(f"  {LIME}{BOLD}── METHODOLOGIES ────────────────────────────────────────{RESET}")
    for m in methods:
        count = sum(1 for p in G.predecessors(m) if G.nodes[p].get("group") == "Paper")
        print(f"  {GREY}[{WHITE}{BOLD}{idx:2d}{RESET}{GREY}]{RESET}  {LIME}{m:<42}{RESET}  {GREY}{count} papers{RESET}")
        all_topics.append(m)
        idx += 1

    print()

    # Authors
    print(f"  {AMBER}{BOLD}── AUTHORS ──────────────────────────────────────────────{RESET}")
    for a in authors:
        count = sum(1 for p in G.successors(a) if G.nodes[p].get("group") == "Paper")
        display = a.replace("_", " ")
        print(f"  {GREY}[{WHITE}{BOLD}{idx:2d}{RESET}{GREY}]{RESET}  {AMBER}{display:<42}{RESET}  {GREY}{count} papers{RESET}")
        all_topics.append(a)
        idx += 1

    print()
    print(f"  {GREY}[{WHITE}{BOLD} 0{RESET}{GREY}]{RESET}  {GREY}Exit{RESET}")
    divider()

    return all_topics


def score_bar(score, max_score=10.0):
    """Render a visual score bar."""
    filled = int((score / max_score) * 20)
    bar = "█" * filled + "░" * (20 - filled)
    return f"{CYAN}{bar}{RESET} {AMBER}{score:.1f}{RESET}"


def print_paper_card(rank, paper, max_score):
    """Print one styled paper recommendation card."""
    divider("·")
    print(f"\n  {BOLD}{WHITE}#{rank:02d}  {paper['title']}{RESET}")
    print(f"\n       Score   {score_bar(paper['score'], max_score)}")
    print(f"       Domain  {PINK}{paper['domain']}{RESET}")
    print(f"       Method  {LIME}{paper['method']}{RESET}")
    print(f"       Author  {AMBER}{paper['author']}{RESET}")
    print(f"       Cited   {GREY}{paper['citations']} times in network{RESET}")

    if paper["reasons"]:
        print(f"\n       {GREY}WHY RECOMMENDED:{RESET}")
        for r in paper["reasons"][:2]:
            print(f"       {GREY}▸ {r}{RESET}")

    if paper["rec"]:
        print(f"\n       {DIM}Action → {paper['rec'][:90]}{'...' if len(paper['rec']) > 90 else ''}{RESET}")
    print()


def show_results(topic, results):
    """Display full recommendation panel for a topic."""
    clear()
    banner()

    group = results[0]["domain"] if results else ""
    color = PINK if "Domain" in str(topic) else LIME

    print(f"  {BOLD}RECOMMENDATIONS FOR:{RESET}  ", end="")
    print(f"{CYAN}{BOLD}{topic.replace('_', ' ')}{RESET}\n")

    if not results:
        print(f"  {RED}No papers found for this topic.{RESET}\n")
        return

    # Summary stats
    direct   = sum(1 for r in results if r["score"] >= 3.0)
    bridging = len(results) - direct
    max_score = results[0]["score"] if results else 1.0

    print(f"  {GREY}Found {WHITE}{BOLD}{len(results)}{RESET} {GREY}papers  ·  "
          f"{WHITE}{BOLD}{direct}{RESET} {GREY}direct match  ·  "
          f"{WHITE}{BOLD}{bridging}{RESET} {GREY}cross-domain bridges{RESET}\n")

    divider("═", CYAN)

    # Split: direct matches vs cross-domain
    direct_papers   = [r for r in results if r["score"] >= 3.0]
    bridging_papers = [r for r in results if r["score"] < 3.0]

    if direct_papers:
        print(f"\n  {CYAN}{BOLD}★  DIRECT MATCHES  (strongly related){RESET}\n")
        for i, paper in enumerate(direct_papers, 1):
            print_paper_card(i, paper, max_score)

    if bridging_papers:
        print(f"\n  {AMBER}{BOLD}⟳  CROSS-DOMAIN BRIDGES  (cite papers in this cluster){RESET}\n")
        for i, paper in enumerate(bridging_papers, 1):
            print_paper_card(len(direct_papers) + i, paper, max_score)

    divider("═", CYAN)
    print(f"\n  {GREY}Tip: Cross-domain bridges are great for finding novel research angles.{RESET}\n")


# ─────────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────────
def main():
    clear()
    banner()
    print(f"  {GREY}Loading network_data.json ...{RESET}")
    G, raw = load_graph("network_data.json")

    node_count  = len([n for n in G.nodes if G.nodes[n]["group"] == "Paper"])
    edge_count  = G.number_of_edges()
    print(f"  {LIME}✓ Graph loaded: {WHITE}{BOLD}{node_count} papers{RESET}{LIME}, "
          f"{WHITE}{BOLD}{edge_count} relationships{RESET}\n")

    while True:
        clear()
        banner()
        all_topics = topic_menu(G)

        try:
            choice = input(f"\n  {WHITE}Enter number (0 to exit): {RESET}").strip()
        except (KeyboardInterrupt, EOFError):
            break

        if choice == "0" or choice == "":
            clear()
            print(f"\n  {CYAN}Thanks for using the Research Recommender. Goodbye!{RESET}\n")
            break

        try:
            idx = int(choice) - 1
            if idx < 0 or idx >= len(all_topics):
                raise ValueError
        except ValueError:
            input(f"  {RED}Invalid choice. Press Enter to retry...{RESET}")
            continue

        topic = all_topics[idx]
        results = recommend(G, topic)

        show_results(topic, results)
        input(f"  {GREY}Press Enter to go back to topic menu...{RESET}")


if __name__ == "__main__":
    main()


![output](output.png)

### Discussion

The lab focused on understanding how knowledge can be represented and processed using Semantic Networks and Natural Language Processing (NLP) techniques. Concepts such as Part-of-Speech (POS) Tagging, Named Entity Recognition (NER), Vector Embeddings, and Knowledge Triples were studied. POS tagging helped identify the grammatical role of words, while NER enabled the extraction of real-world entities such as people, organizations, and locations. Vector embeddings demonstrated how textual data can be converted into numerical representations for machine learning applications. The construction of knowledge triples and semantic networks showed how relationships between concepts can be organized and utilized for tasks such as information retrieval and recommendation systems.

### Conclusion

This lab provided practical knowledge of semantic representation and NLP techniques used in modern AI systems. By exploring POS tagging, NER, vector embeddings, and knowledge triples, a better understanding was gained of how computers interpret, organize, and analyze textual information. The concepts learned in this lab form the foundation for advanced applications such as knowledge graphs, search engines, chatbots, and recommendation systems.